**PROJECT : Movies Recommendation Engine**

#Import important libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#Load Data set

In [ ]:
df=pd.read_csv("/movies.csv")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB


#Clean dataset so no need of Data Cleaning

In [ ]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
print(df.columns.tolist())

['movieId', 'title', 'genres']


In [ ]:
df.nunique()

,0
movieId,10329
title,10327
genres,938


In [ ]:
print("Rows count : ",df.shape[0])
print("Columns count : ",df.shape[1])

Rows count :  10329
Columns count :  3


In [ ]:
df.tail()

,movieId,title,genres
10324,146684,Cosmic Scrat-tastrophe (2015),Animation|Children|Comedy
10325,146878,Le Grand Restaurant (1966),Comedy
10326,148238,A Very Murray Christmas (2015),Comedy
10327,148626,The Big Short (2015),Drama
10328,149532,Marco Polo: One Hundred Eyes (2015),(no genres listed)


#Training & Testing Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

In [ ]:
df.columns=df.columns.str.strip()

In [ ]:
X = df['movieId'].values.reshape(-1, 1)

In [ ]:
y = df['genres']

In [ ]:
# Train, Test Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=200)

In [ ]:
X_test

array([[27838],
       [ 2474],
       [ 5088],
       ...,
       [  121],
       [   17],
       [53550]])

In [ ]:
y_test

,genres
6393,Drama|Thriller
1979,Drama
3952,Comedy|Crime|Drama
6449,Drama|Sci-Fi
1930,Action|Adventure|Drama
...,...
5628,Comedy|Fantasy|Thriller
7680,Comedy
107,Drama
16,Drama|Romance


#Convert genres into features

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
df['genres'] = df['genres'].apply(lambda x: x.split('|'))

In [ ]:
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(df['genres'])

In [ ]:
# Convert to DataFrame
genre_df = pd.DataFrame(genre_features, columns=mlb.classes_)

In [ ]:
# Merge back
df = pd.concat([df, genre_df], axis=1)

In [ ]:
df.drop('genres', axis=1, inplace=True)
print(df.head(10))

   movieId                               title  (no genres listed)  Action  \
0        1                    Toy Story (1995)                   0       0   
1        2                      Jumanji (1995)                   0       0   
2        3             Grumpier Old Men (1995)                   0       0   
3        4            Waiting to Exhale (1995)                   0       0   
4        5  Father of the Bride Part II (1995)                   0       0   
5        6                         Heat (1995)                   0       1   
6        7                      Sabrina (1995)                   0       0   
7        8                 Tom and Huck (1995)                   0       0   
8        9                 Sudden Death (1995)                   0       1   
9       10                    GoldenEye (1995)                   0       1   

   Adventure  Animation  Children  Comedy  Crime  Documentary  ...  Film-Noir  \
0          1          1         1       1      0            

#Converting every genres into a numerical format using tf idf

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df=pd.read_csv("/movies.csv")
# Replace | with space
df['genres'] = df['genres'].str.replace('|', ' ', regex=False)
# Initialize TF-IDF
tfidf = TfidfVectorizer()
# Fit and transform
tfidf_matrix = tfidf.fit_transform(df['genres'])

In [ ]:
# Convert sparse matrix to DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=tfidf.get_feature_names_out())
print(df.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Children Fantasy  
2                               Comedy Romance  
3                         Comedy Drama Romance  
4                                       Comedy  


In [ ]:
df.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action Crime Thriller
6,7,Sabrina (1995),Comedy Romance
7,8,Tom and Huck (1995),Adventure Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action Adventure Thriller


#Calculating Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Calculate similarity
similarity_matrix = cosine_similarity(tfidf_matrix)
print(similarity_matrix.shape)

(10329, 10329)


In [ ]:
movie_index = 10 # movie_index = 10 is just only taken as example
similar_movies = list(enumerate(similarity_matrix[movie_index]))
similar_movies = sorted(similar_movies, key=lambda x: x[1], reverse=True)

In [ ]:
for i in similar_movies[1:6]:
    print("Movie Index:", i[0], "Similarity Score:", i[1])

Movie Index: 10 Similarity Score: 1.0
Movie Index: 48 Similarity Score: 1.0
Movie Index: 53 Similarity Score: 1.0
Movie Index: 86 Similarity Score: 1.0
Movie Index: 168 Similarity Score: 1.0


#Recommendation Function

In [ ]:
def recommend_movies(title, top_n=5):
    if title not in df['title'].values:
        return "Movie not found in dataset."

    idx = df[df['title'] == title].index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    # Sort movies based on similarity score
    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)

    top_movies = similarity_scores[1:top_n+1]

    print(f"\nMovies similar to '{title}':\n")

    for movie in top_movies:
        print(df.iloc[movie[0]]['title'],
              " | Similarity Score:", round(movie[1], 3))

In [ ]:
recommend_movies("Interstellar (2014)", top_n=10)


Movies similar to 'Interstellar (2014)':

Cloud Atlas (2012)  | Similarity Score: 0.974
Transcendence (2014)  | Similarity Score: 0.974
Contagion (2011)  | Similarity Score: 0.942
Gravity (2013)  | Similarity Score: 0.932
The Amazing Spider-Man 2 (2014)  | Similarity Score: 0.932
Edge of Tomorrow (2014)  | Similarity Score: 0.932
Day the Earth Stood Still, The (2008)  | Similarity Score: 0.92
Real Steel (2011)  | Similarity Score: 0.911
Elysium (2013)  | Similarity Score: 0.911
Men in Black III (M.III.B.) (M.I.B.³) (2012)  | Similarity Score: 0.9


#Test Recommenation Engine

In [ ]:
recommend_movies("Toy Story (1995)")


Movies similar to 'Toy Story (1995)':

Antz (1998)  | Similarity Score: 1.0
Toy Story 2 (1999)  | Similarity Score: 1.0
Adventures of Rocky and Bullwinkle, The (2000)  | Similarity Score: 1.0
Emperor's New Groove, The (2000)  | Similarity Score: 1.0
Monsters, Inc. (2001)  | Similarity Score: 1.0


In [ ]:
recommend_movies("The Dark Knight Rises (2012)")

'Movie not found in dataset.'

#Final Recommendation Engine Model

In [ ]:
enter_movie_name=input("Enter Movie Name : ")
recommend_movies(enter_movie_name,top_n=30)

Enter Movie Name : Border


'Movie not found in dataset.'